In [2]:
import os
import json
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from openai import OpenAI
import ollama

In [3]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [4]:
DATA_PATH = "../data"

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 9


In [5]:
for i, doc in enumerate(documents):
    print(i, len(doc.page_content))

0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0


In [5]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'page': 0}


In [6]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        docs = loader.load()
        
        # DEBUG
        print(file, "pages:", len(docs))
        
        documents.extend(docs)

doc1.pdf pages: 2
doc2.pdf pages: 4
doc3.pdf pages: 3


In [7]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'page': 0}


In [8]:
from langchain_community.document_loaders import PyMuPDFLoader

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 9


In [9]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'file_path': '../data\\doc1.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': 'doc1.md - Google Docs', 'author': 'Subrata Das', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Microsoft: Print To PDF', 'creationDate': "D:20260323230807+05'30'", 'modDate': "D:20260323230807+05'30'", 'trapped': ''}


In [10]:
print(documents[0].page_content[:300])

In [6]:
import os
from pdf2image import convert_from_path
import pytesseract
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Tesseract path
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path
POPPLER_PATH = r"C:\Users\subra\Downloads\poppler-25.12.0\Library\bin"
DATA_PATH = "../data"

documents = []

# 🔹 STEP 1: PDF → Images → OCR → Documents
for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(DATA_PATH, file)

        print(f"Processing: {file}")

        # Convert PDF to images
        images = convert_from_path(
            pdf_path,
            poppler_path=POPPLER_PATH  # remove if PATH is set
        )

        for i, image in enumerate(images):
            text = pytesseract.image_to_string(image)

            import re

            text = re.sub(r"Q:.*?A:", "", text)  # remove Q&A patterns
            # Clean text (important)
            text = " ".join(text.split())

            if text:  # avoid empty pages
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "page": i
                        }
                    )
                )

print("Total documents:", len(documents))

Processing: doc1.pdf
Processing: doc2.pdf
Processing: doc3.pdf
Total documents: 9


In [7]:
print("\nSample extracted text:\n")
print(documents[0].page_content[:500]) #OCR working :) image->text conversion successful!


Sample extracted text:

INDECIMAL — Company Overview & Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building “confidence” through commitment, not just con


In [8]:
import re

def clean_text(text):
    # 1. Normalize whitespace
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    # 🔥 2. REMOVE Q&A PATTERNS (CRITICAL FIX)
    text = re.sub(r"Q:.*?A:", "", text, flags=re.IGNORECASE)

    # 🔥 3. Remove leftover Q: or A:
    text = re.sub(r"\bQ:\s*", "", text)
    text = re.sub(r"\bA:\s*", "", text)

    # 4. Fix OCR mistakes
    replacements = {
        "saft": "sqft",
        "saqft": "sqft",
        "sq ft": "sqft",
        "rs.": "Rs",
        "₹": "Rs ",
        "=": "",
        "%": "",
        "—": "-",
        "–": "-",
    }

    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    # 🔥 5. Remove weird currency/symbol noise like ¥, §
    text = re.sub(r"[¥§]", "", text)

    # 🔥 6. Keep useful characters, remove junk
    text = re.sub(r"[^\w\s\.,:/\-\(\)]", "", text)

    # 🔥 7. Fix spacing around punctuation
    text = re.sub(r"\s+([.,])", r"\1", text)

    # 🔥 8. Remove repeated dashes or dots
    text = re.sub(r"-{2,}", "-", text)
    text = re.sub(r"\.{2,}", ".", text)

    return text.strip()

In [9]:
cleaned_documents = []
for doc in documents:
    cleaned_text = clean_text(doc.page_content)
    if cleaned_text:  # avoid empty text
        cleaned_documents.append(
            Document(
                page_content=cleaned_text,
                metadata=doc.metadata
            )
        )

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(cleaned_documents)

print("\nTotal chunks:", len(chunks))


Total chunks: 18


In [11]:
print("Sample chunk:\n")
print(chunks[0].page_content)

Sample chunk:

INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emphasizes clarity and trust across the construction and ownership journey. 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. Best and Competitive Pricing - Fair pricing with no hidden charges


In [12]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
vectorstore = FAISS.from_documents(chunks, embedding)

In [15]:
vectorstore.save_local("../faiss_index")
print("FAISS index saved")

FAISS index saved


In [16]:
vectorstore = FAISS.load_local(
    "../faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

In [17]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 7})

In [18]:
from langchain_community.llms import Ollama
from langchain_openai import ChatOpenAI
llm_openrouter = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model="meta-llama/llama-3.1-8b-instruct",
    temperature=0
)
llm_local = Ollama(model="phi3",temperature=0)

In [20]:
#improving retrieval by reranker
from sentence_transformers import CrossEncoder
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
def rerank_docs(docs, query):
    
    # Create query-doc pairs
    pairs = [[query, doc.page_content] for doc in docs]
    
    # Get relevance scores
    scores = reranker_model.predict(pairs)
    
    # Sort by score (descending)
    ranked_docs = [doc for _, doc in sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )]
    
    return ranked_docs

In [22]:
import time
def run_rag(query,llm,model_name="unknown"):
    
    # 🔹 Step 1: Retrieve (k=7)
    docs = retriever.get_relevant_documents(query)
    
    # 🔹 Step 2: Rerank (🔥 cross-encoder)
    docs = rerank_docs(docs, query)
    
    # 🔹 Step 3: Take top 3
    docs = docs[:3]
    
    # 🔹 Step 4: Build context
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 🔹 Step 5: Prompt
    prompt = f"""
    You are a strict QA assistant.

    Answer ONLY the given question using the context below.

    Rules:
    - Do NOT add any information not present in context
    - If information is not present in context, say: "Not found in context"
    - Do NOT guess or assume
    - Keep answer factual and short
    -Focus on pricing-related information when question is about pricing
    -Do NOT give partial answers
    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    start= time.time()
    response = llm.invoke(prompt)
    end = time.time()

    return {
        "model": model_name,
        "question": query,
        "answer": response.content if hasattr(response, "content") else str(response),
        "latency": end - start,
        "sources": [doc.page_content for doc in docs]
    }

In [23]:
def debug_pipeline(query):
    
    print("\n================ QUERY ================\n")
    print(query)
    
    # 🔹 Step 1: Retrieve (FAISS)
    retrieved_docs = retriever.get_relevant_documents(query)
    
    print("\n=========== RETRIEVED (FAISS) ===========\n")
    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- Retrieved {i+1} ---")
        print(doc.page_content[:200])
        print("Source:", doc.metadata)
    
    # 🔹 Step 2: Rerank
    reranked_docs = rerank_docs(retrieved_docs, query)
    
    print("\n=========== RERANKED (CROSS-ENCODER) ===========\n")
    for i, doc in enumerate(reranked_docs):
        print(f"\n--- Reranked {i+1} ---")
        print(doc.page_content[:200])
        print("Source:", doc.metadata)
    
    return reranked_docs

In [ ]:
debug_pipeline("What is indecimal pricing?")    


================ QUERY ================

What is indecimal pricing?

=========== RETRIEVED (FAISS) ===========


--- Retrieved 1 ---
INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecim
Source: {'source': 'doc1.pdf', 'page': 0}

--- Retrieved 2 ---
INDECIMAL - Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project Management, Al Assistant Knowledge Base Last Updated: 2025-12
Source: {'source': 'doc3.pdf', 'page': 0}

--- Retrieved 3 ---
. Best and Competitive Pricing - Fair pricing with no hidden charges. Quality Assurance (445 checks) - Strict quality control at every construction stage. Stage-Based Contractor Payments - Payments re
Source: {'source': 'doc1.pdf', 'page': 0}

--- Retrieved 4 ---
INDECIMAL - Package Comparison  Specification Wallets (Internal Reference)

c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(



=========== RERANKED (CROSS-ENCODER) ===========


--- Reranked 1 ---
INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecim
Source: {'source': 'doc1.pdf', 'page': 0}

--- Reranked 2 ---
INDECIMAL - Package Comparison  Specification Wallets (Internal Reference) Version: 1.0 Audience: Sales, Estimation, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Package Pricing (Indicative
Source: {'source': 'doc2.pdf', 'page': 0}

--- Reranked 3 ---
. Best and Competitive Pricing - Fair pricing with no hidden charges. Quality Assurance (445 checks) - Strict quality control at every construction stage. Stage-Based Contractor Payments - Payments re
Source: {'source': 'doc1.pdf', 'page': 0}

--- Reranked 4 ---
INDECIMAL - Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project M

[Document(metadata={'source': 'doc1.pdf', 'page': 0}, page_content='INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emphasizes clarity and trust across the construction and ownership journey. 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. Best and Competitive Pricing - Fair pricing with no hidden charges'),
 Document(metadata={'source': 'doc2.pdf', 'page': 0}, page_content='INDECIMAL - Package Comparison  Specification Wallets (Internal Reference)

In [28]:
query="What is the pricing structure used by Indecimal?"
res_local = run_rag(query, llm_local, model_name="phi3-local")
res_openrouter = run_rag(query, llm_openrouter, model_name="llama3.1-openrouter")
print("🔹 QUESTION:", query)

print("\n--- LOCAL (phi3) ---")
print("Answer:", res_local["answer"])
print("Latency:", res_local["latency"])

print("\n--- OPENROUTER (llama3) ---")
print("Answer:", res_openrouter["answer"])
print("Latency:", res_openrouter["latency"])

🔹 QUESTION: What is the pricing structure used by Indecimal?

--- LOCAL (phi3) ---
Answer: Indecimal uses a package pricing structure per square foot, with options for Essential at 1,851 /sqft and Premier at 1,995 /sqft. Additionally, they offer Infinia at 2,250 /sqft and Pinnacle packages starting from 2,450 /sqft per square foot.
Latency: 7.141855478286743

--- OPENROUTER (llama3) ---
Answer: Per-sqft package rates (inclusive of GST).
Latency: 1.8045837879180908


In [ ]:
debug_pipeline("What are the package rates?")    


================ QUERY ================

What are the package rates?

=========== RETRIEVED (FAISS) ===========


--- Retrieved 1 ---
INDECIMAL - Package Comparison  Specification Wallets (Internal Reference) Version: 1.0 Audience: Sales, Estimation, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Package Pricing (Indicative
Source: {'source': 'doc2.pdf', 'page': 0}

--- Retrieved 2 ---
. Block Work - Solid concrete blocks 6 (external)  4 (internal) - Pricing guidance shown as: - 6: up to 40 (/- 3) per block
Source: {'source': 'doc2.pdf', 'page': 0}

--- Retrieved 3 ---
- 4: up to 33 (/- 3) per block RCC Mix - M20 or M25 (RCC design mix as advised by the structural engineer) Ceiling Height - Floor-to-floor height: 10 ft (across packages) 3) Kitchen (Wallet Allowances
Source: {'source': 'doc2.pdf', 'page': 1}

--- Retrieved 4 ---
Sanitary  CP Fittings (per 1000 sqft) Essential: up to 32,000 (Cera, Hindware / equivalent) Premier: up to 50,000 (Parryware / equivalent) Infinia: up

[Document(metadata={'source': 'doc2.pdf', 'page': 0}, page_content='INDECIMAL - Package Comparison  Specification Wallets (Internal Reference) Version: 1.0 Audience: Sales, Estimation, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Package Pricing (Indicative / Per Sqft) These are shown as per-sqft package rates (inclusive of GST) on the public comparison page: - Essential: 1,851 /sqft (incl. GST) - Premier (Most Popular): 1,995 /sqft (incl. GST) - - Infinia: 2,250 /sqft (incl. GST) - Pinnacle: 2,450 /sqft (incl'),
 Document(metadata={'source': 'doc2.pdf', 'page': 1}, page_content='- 4: up to 33 (/- 3) per block RCC Mix - M20 or M25 (RCC design mix as advised by the structural engineer) Ceiling Height - Floor-to-floor height: 10 ft (across packages) 3) Kitchen (Wallet Allowances) Note: The page states fittings can be customized at cost'),
 Document(metadata={'source': 'doc2.pdf', 'page': 0}, page_content='. Block Work - Solid concrete blocks 6 (external)  4 (internal) - Pricin

In [ ]:
eval_data = [
    {
        "question": "What is the pricing structure used by Indecimal?",
        "ground_truth": "Package-based pricing per sqft including GST with transparent cost structure"
    },
    {
        "question": "What are the package rates offered by Indecimal?",
        "ground_truth": "₹1851, ₹1995, ₹2250, and ₹2450 per sqft for different packages"
    },
    {
        "question": "What is the Essential package rate per sqft?",
        "ground_truth": "₹1851 per sqft"
    },
    {
        "question": "What is the price difference between Premier and Infinia packages?",
        "ground_truth": "Premier costs ₹1995 per sqft while Infinia costs ₹2250 per sqft"
    },
    {
        "question": "How does Indecimal ensure transparent pricing?",
        "ground_truth": "By providing detailed cost plans with no hidden charges and clear package pricing"
    },
    {
        "question": "What payment system does Indecimal use during construction?",
        "ground_truth": "Escrow-based stage-wise payment system"
    },
    {
        "question": "How are contractor payments controlled in Indecimal projects?",
        "ground_truth": "Payments are released only after verification of completed construction stages"
    },
    {
        "question": "What customer protection mechanisms are provided by Indecimal?",
        "ground_truth": "Escrow payments, stage verification, and quality control processes"
    },
    {
        "question": "What services does Indecimal provide from start to finish?",
        "ground_truth": "End-to-end home construction support from initial consultation to project handover"
    },
    {
        "question": "What tools does Indecimal provide for customers to monitor construction progress?",
        "ground_truth": "Through a real-time project tracking dashboard with visibility into progress and metrics"
    }
]

In [ ]:
results = []

for item in eval_data:
    question = item["question"]
    
    #Run BOTH models
    res_local = run_rag(question, llm_local, "phi3-local")
    res_openrouter = run_rag(question, llm_openrouter, "llama3-openrouter")
    
    results.append({
        "question": question,
        
        #Answers
        "local_answer": res_local["answer"],
        "openrouter_answer": res_openrouter["answer"],
        
        #Latency
        "local_latency": res_local["latency"],
        "openrouter_latency": res_openrouter["latency"],
        
        #Context (same retrieval)
        "contexts": res_local["sources"],
        
        #Ground truth
        "ground_truth": item["ground_truth"]
    })

In [ ]:
results

[{'question': 'What is the pricing structure used by Indecimal?',
  'local_answer': 'Indecimal uses a package pricing structure per square foot, with options for Essential at 1,851 /sqft and Premier at 1,995 /sqft. Additionally, they offer Infinia at 2,250 /sqft and Pinnacle packages starting from 2,450 /sqft.',
  'openrouter_answer': 'Per-sqft package rates (inclusive of GST).',
  'local_latency': 7.629330158233643,
  'openrouter_latency': 2.86944317817688,
  'contexts': ['INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emp

In [ ]:
import json

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("✅ results.json saved")

✅ results.json saved
